# Seguindo Estratégias do EDA_2023

Justificação das Transformações de Dados (EDA)
1. Criação da flag Tem_Ingles antes da imputação de zeros

Porquê: Em Machine Learning e análise estatística, existe uma diferença abismal entre "o aluno teve nota zero na prova" e "o aluno não teve a disciplina". Se imputarmos diretamente o valor 0 nos nulos (NaN), o modelo interpretará que todos esses alunos falharam redondamente no Inglês. Ao criar a variável categórica Tem_Ingles (1 para sim, 0 para não), preservamos a realidade estrutural do currículo da ONG, ensinando ao modelo que a ausência de nota se deve à não obrigatoriedade da disciplina naquelas fases específicas.

2. Recriação dos Rankings (CG_2023, CF_2023 e CT_2023)

Porquê: As colunas originais de Classificação Geral (CG), Classificação por Fase (CF) e Classificação por Turma (CT) estavam vazias. No entanto, o ranking nada mais é do que a ordenação da nota final. Ao reconstruir estas métricas com base no INDE 2023, recuperamos atributos valiosos de posicionamento relativo. Isto permite avaliar o desempenho de um aluno não apenas pela sua nota absoluta, mas pela sua posição competitiva dentro do seu grupo de pares. (Nota: as colunas foram ajustadas para o sufixo 2023 para manter a coerência do ano analisado).

3. Eliminação de colunas 100% nulas e renomeação de variáveis reais

Porquê: A exclusão das colunas de recomendações (Rec Av1, Rec Psicologia, etc.), destaques e duplicados vazios (INDE 23, Pedra 23) é um passo fundamental de redução de dimensionalidade. Variáveis com 100% de valores nulos não contêm variância e apenas introduzem "ruído" computacional e visual. Em simultâneo, renomear as colunas válidas (ex: INDE 2023 para INDE_23) elimina artefactos de exportação do sistema e garante a integridade nas futuras cruzas de dados.

4. Definição de Threshold (corte) para o Indicador IPV

Porquê: O "Ponto de Virada" não é apenas uma nota matemática, mas um marco de maturidade psicológica e pedagógica. Como a base de 2023 não trouxe esta classificação preenchida (Sim/Não), estabelecer um limite de corte baseado na distribuição estatística histórica do IPV (Indicador de Ponto de Virada) permite automatizar esta classificação. Traduzimos assim uma variável contínua (nota) numa regra de negócio binária e acionável.

5. Critério de Indicação para Bolsa baseado no INDE

Porquê: A atribuição de bolsas é estritamente meritocrática, sendo impulsionada pelo Desempenho Académico (IDA) e Engajamento (IEG), que juntos compõem a maior fatia do INDE. Utilizar os níveis hierárquicos mais altos da ONG (Pedras Topázio e Ametista) como gatilho lógico para a coluna Indicado simula com precisão as regras de seleção do mundo real, identificando de forma automática os talentos de alto rendimento.

6. Desmembramento da "Fase Ideal" em Fase e Série Escolar

Porquê: A coluna original agregava duas informações distintas: o nível interno na metodologia da ONG (Fase) e o ano letivo no sistema de ensino tradicional (Série Escolar). Separar estes dados aumenta a granularidade da análise. Passa a ser possível cruzar a "Série Escolar" com a "Idade" para calcular taxas de defasagem escolar externa de forma totalmente independente da progressão interna do aluno na associação.

7. Mapeamento e Padronização do Cabeçalho de Colunas

Porquê: É uma das melhores práticas em Engenharia de Dados. Padronizar os nomes (letras minúsculas, substituição de espaços por underscores, nomenclatura curta) previne erros de sintaxe durante a programação. Mais importante ainda, garante que o dataset de 2023 tenha a mesma exata estrutura dos anos anteriores (como 2022), permitindo concatenações (pd.concat) perfeitas para análises históricas de longo prazo sem duplicação acidental de colunas.

# Carregando DF

In [443]:
import pandas as pd

# O parâmetro sheet_name=None diz ao Pandas para carregar TUDO
todas_as_folhas = pd.read_excel('data/base_dados_2024.xlsx', sheet_name=None)

# Para ver os nomes de todas as folhas que foram carregadas:
print(todas_as_folhas.keys())

df = todas_as_folhas['PEDE2023']

dict_keys(['PEDE2022', 'PEDE2023', 'PEDE2024'])


# 1. Inglês

In [444]:
# Criando a flag (0 ou 1) para o modelo futuro
df['Tem_Ingles'] = df['Ing'].apply(lambda x: 0 if pd.isna(x) else 1)

# Preenchendo com 0 apenas para viabilizar as contas de ranking/média
df['Ing'] = df['Ing'].fillna(0)

print(f"Alunos com Inglês: {df['Tem_Ingles'].sum()}")
print(f"Alunos sem Inglês: {len(df) - df['Tem_Ingles'].sum()}")

Alunos com Inglês: 334
Alunos sem Inglês: 680


# 2. Criação dos Rankings 2023 (Cg, Cf, Ct)

In [445]:
# 1. Ranking Geral (Cg)
df['Cg'] = df['INDE 2023'].rank(ascending=False, method='min')

# 2. Ranking por Fase (Cf)
df['Cf'] = df.groupby('Fase')['INDE 2023'].rank(ascending=False, method='min')

# 3. Ranking por Turma (Ct)
df['Ct'] = df.groupby('Turma')['INDE 2023'].rank(ascending=False, method='min')

# Visualizando o resultado para os TOP 5 do Geral
print("\nTop 5 Alunos Geral (2023):")
display(df[['Cg', 'Cf', 'Ct', 'Fase', 'Turma', 'INDE 2023']].sort_values(by='Cg').head())


Top 5 Alunos Geral (2023):


,Cg,Cf,Ct,Fase,Turma,INDE 2023
107,1.0,1.0,1.0,ALFA,ALFA K - G0/G1,9.37120
116,2.0,2.0,2.0,ALFA,ALFA K - G0/G1,9.32370
0,3.0,3.0,1.0,ALFA,ALFA A - G0/G1,9.31095
156,4.0,4.0,1.0,ALFA,ALFA O - G2/G3,9.29020
8,5.0,5.0,2.0,ALFA,ALFA A - G0/G1,9.25870


# 3. Apagar colunas desnecessarias





In [446]:
colunas_remover = [
    'INDE 23', 'Pedra 23', 'Rec Av1', 'Rec Av2', 'Rec Av3', 'Rec Av4', 'Rec Psicologia', 
    'Destaque IEG', 'Destaque IDA', 'Destaque IPV', 'Destaque IPV.1',
    'Indicado', 'Atingiu PV', 'Avaliador3', 'Avaliador4' , 'Avaliador1' , 'Avaliador2', 'Nº Av'
]
df.drop(columns=colunas_remover, inplace=True, errors='ignore')

# 4. Renomear Pedra 2023 para Pedra_23  e  renomear INDE 2023 para INDE_23

In [447]:
# Renomeando as colunas oficiais para o padrão de 2023
df.rename(columns={
    'Pedra 2023': 'Pedra_23',
    'INDE 2023': 'INDE_23'
}, inplace=True)

# Verificando como ficou o cabeçalho
print("Colunas atualizadas:")
print(df[['INDE_23', 'Pedra_23']].head())

Colunas atualizadas:
   INDE_23  Pedra_23
0  9.31095   Topázio
1  8.22120   Topázio
2  5.92975   Quartzo
3  7.03400  Ametista
4  8.15520   Topázio


# 5. Ponto de Virada (PV) e Indicação para Bolsa.

In [448]:
#  Cálculo do INDE_23 (Pesos oficiais da metodologia PEDE)
# Os pesos seguem a importância atribuída no relatório:
# IDA (20%), IEG (20%), IPV (20%) e os demais com 10% cada.
df['INDE_23'] = (
    df['IAN'] * 0.1 +
    df['IDA'] * 0.2 +
    df['IEG'] * 0.2 +
    df['IAA'] * 0.1 +
    df['IPS'] * 0.1 +
    df['IPP'] * 0.1 +
    df['IPV'] * 0.2
)

#  Definição do status "Atingiu PV" (Ponto de Virada)
# O Ponto de Virada é um marco de maturidade. Segundo o relatório, 
# alunos com IPV elevado atingem este status. 
# O critério sugerido é a nota de corte 8.0 (conforme os tiers de alto desempenho).
df['Atingiu_PV'] = df['IPV'].apply(lambda x: 'Sim' if x >= 8.0 else 'Não')

# Definição do status "Indicado" (Bolsa de Estudos)
# A indicação para bolsa é baseada no mérito (Pedras Ametista ou Topázio).
# Geralmente, isso corresponde a um INDE superior a 6.8 (início da Ametista).
def definir_indicado(row):
    # Critério: Estar nos níveis de desempenho Ametista ou Topázio
    if row['INDE_23'] >= 6.81:
        return 'Sim'
    return 'Não'

df['Indicado_Bolsa'] = df.apply(definir_indicado, axis=1)

# Verificação dos resultados
print(df[['RA', 'INDE_23', 'Atingiu_PV', 'Indicado_Bolsa']].head())

       RA  INDE_23 Atingiu_PV Indicado_Bolsa
0  RA-861  9.31075        Sim            Sim
1  RA-862  8.23100        Sim            Sim
2  RA-863  5.93975        Não            Não
3  RA-864  7.04400        Sim            Sim
4  RA-865  8.15500        Não            Sim


# 6. Desmembramento da Fase Ideal e Série Escolar

In [449]:
# O nome na base 2023 é 'Fase Ideal' (com maiúsculas)
df[['fase_limpa', 'serie_escolar']] = df['Fase Ideal'].str.split(r' \(', expand=True)

# Limpeza e tratamento de strings
df['serie_escolar'] = df['serie_escolar'].str.replace(')', '', regex=False).str.strip()
df['fase_limpa'] = df['fase_limpa'].str.strip()

# Tratamento para o grupo de alfabetização (ALFA)
df.loc[df['fase_limpa'].str.contains('ALFA', na=False), 'fase_limpa'] = 'ALFA'
df.loc[df['fase_limpa'] == 'ALFA', 'serie_escolar'] = 'Alfabetização'

print("--- Distribuição de Fases e Séries 2023 ---")
print(df[['fase_limpa', 'serie_escolar']].value_counts())

--- Distribuição de Fases e Séries 2023 ---
fase_limpa  serie_escolar 
Fase 2      5° e 6° ano       246
Fase 3      7° e 8° ano       204
ALFA        Alfabetização     116
Fase 1      3° e 4° ano       108
Fase 4      9° ano             96
Fase 8      Universitários     78
Fase 5      1° EM              65
Fase 6      2° EM              55
Fase 7      3° EM              46
Name: count, dtype: int64


# 7.  Auditar e provar a regra de negócio por engenharia reversa

Objetivo: Validar matematicamente a fórmula de cálculo do INDE_23 (Índice de Desenvolvimento Educacional) utilizando Machine Learning, garantindo que os pesos dos indicadores de 2023 se mantêm consistentes com a metodologia histórica da Passos Mágicos.

In [450]:
from sklearn.linear_model import LinearRegression

# Criar um dataframe temporário só com quem tem todas as notas
indicadores = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV']
df_pesos = df.dropna(subset=['INDE_23'] + indicadores).copy()

# Remover a Fase 8 (Universitários), pois eles costumam ter uma fórmula à parte
df_pesos = df_pesos[~df_pesos['fase_limpa'].astype(str).str.contains('8')]

# Separar as Variáveis (X = Indicadores, y = Nota Final)
X = df_pesos[indicadores]
y = df_pesos['INDE_23']

# Aplicar o Algoritmo (fit_intercept=False força a fórmula a partir de zero, sem viés)
modelo = LinearRegression(fit_intercept=False)
modelo.fit(X, y)

# Visualizar a Mágica: Os coeficientes do modelo são os pesos exatos!
pesos_descobertos = pd.DataFrame({
    'Indicador': X.columns,
    'Peso_Matematico': modelo.coef_,
    'Peso_Arredondado': modelo.coef_.round(1) 
})

print(pesos_descobertos)
print(f"\nSoma total dos pesos arredondados: {pesos_descobertos['Peso_Arredondado'].sum():.2f} (Deveria ser 1.00)")

  Indicador  Peso_Matematico  Peso_Arredondado
0       IAN              0.1               0.1
1       IDA              0.2               0.2
2       IEG              0.2               0.2
3       IAA              0.1               0.1
4       IPS              0.1               0.1
5       IPP              0.1               0.1
6       IPV              0.2               0.2

Soma total dos pesos arredondados: 1.00 (Deveria ser 1.00)


# 8. Mapeamento e Padronização de Colunas

In [451]:
# --- MAPEAMENTO 2023 ---
map_23 = {
    'RA': 'ra', 'Nome Anonimizado': 'nome', 'Gênero': 'genero', 'Idade': 'idade',
    'Ano ingresso': 'ano_ingresso', 'fase_limpa': 'fase ideal', 'Turma': 'turma',
    'serie_escolar': 'serie_escolar', 'ponto_virada': 'ponto_virada', 
    'INDE_23': 'inde',        
    'Pedra_23': 'pedra',     
    'INDE 22': 'inde_2022',
    'Pedra 22': 'pedra_2022',
    'Pedra 21': 'pedra_2021',
    'Pedra 20': 'pedra_2020',
    'nota_mat': 'nota_mat', 'nota_port': 'nota_port', 'nota_ing': 'nota_ing', 
    'Defasagem': 'defas', 
    'Instituição de ensino': 'instituicao_de_ensino', 
}
# Aplicar renomeação e padronização final
df_final = df.rename(columns=map_23)


In [452]:
# =====================================================================
# AUDITORIA FINAL DE VALORES NULOS
# =====================================================================
print("\n" + "="*50)
print("--- VERIFICAÇÃO DE DADOS FALTANTES ---")

# Conta os nulos por coluna
nulos_por_coluna = df_final.isnull().sum()

# Filtra apenas as colunas que têm mais de 0 nulos e ordena da maior para a menor
colunas_com_nulos = nulos_por_coluna[nulos_por_coluna >= 0].sort_values(ascending=False)

# Verifica se o filtro encontrou alguma coisa
if colunas_com_nulos.empty:
    print(" SUCESSO! O dataset de 2023 está 100% limpo, sem nenhum valor nulo.")
else:
    print("ATENÇÃO! As seguintes colunas ainda possuem valores nulos:\n")
    
    # Monta uma tabela formatada para exibir o Nome da Coluna, a Quantidade e a %
    tabela_nulos = pd.DataFrame({
        'Quantidade de Nulos': colunas_com_nulos,
        'Porcentagem (%)': (colunas_com_nulos / len(df_final)) * 100
    })
    
    # Exibe a tabela com 2 casas decimais
    print(tabela_nulos.round(2))

print("="*50 + "\n")


--- VERIFICAÇÃO DE DADOS FALTANTES ---
ATENÇÃO! As seguintes colunas ainda possuem valores nulos:

                       Quantidade de Nulos  Porcentagem (%)
pedra_2020                             774            76.33
pedra_2021                             679            66.96
inde_2022                              414            40.83
pedra_2022                             414            40.83
pedra                                   83             8.19
inde                                    83             8.19
Ct                                      83             8.19
Cf                                      83             8.19
Cg                                      83             8.19
Mat                                     77             7.59
Por                                     77             7.59
IDA                                     77             7.59
IPV                                     76             7.50
IPP                                     76             7.50


Por enquanto vamos deixar os numeros nulos de Pedra e Inde, para posteriormente quando juntar os CSV tratar esses dados.

In [453]:
# Normalização residual (lower case e underscore)
df_final.columns = [col.lower().replace(' ', '_') for col in df_final.columns]

# Adicionar flag de ano para o EDA histórico
df_final['ano_referencia'] = 2023

print("\n--- Colunas Padronizadas para 2023 ---")
print(df_final.columns.tolist())

# Salvar versão final
df_final.to_csv('pede_2023_final.csv', index=False, encoding='utf-8-sig')
print("\n Base 2023 Padronizada e Salva!")


--- Colunas Padronizadas para 2023 ---
['ra', 'fase', 'inde', 'pedra', 'turma', 'nome', 'data_de_nasc', 'idade', 'genero', 'ano_ingresso', 'instituicao_de_ensino', 'pedra_2020', 'pedra_2021', 'pedra_2022', 'inde_2022', 'cg', 'cf', 'ct', 'iaa', 'ieg', 'ips', 'ipp', 'ida', 'mat', 'por', 'ing', 'ipv', 'ian', 'fase_ideal', 'defas', 'tem_ingles', 'atingiu_pv', 'indicado_bolsa', 'fase_ideal', 'serie_escolar', 'ano_referencia']

 Base 2023 Padronizada e Salva!
